# HAST Zeitreihen: Daten laden & EDA

**Zweck:** **Zweck:** Lädt die echten UEZ-Zähler-Zeitreihen aus `data/raw/` (CSV- und ODS-Exporte), führt alle Dateien pro Zähler zusammen und berechnet eine Feature-Tabelle pro HAST — als Grundlage für explorative Analyse und später Clustering.


**Eigenständig:** Dieses Notebook braucht nur `data/raw/` und Standard-Bibliotheken (pandas, numpy, matplotlib, odfpy für die ODS-Datei) — keine Abhängigkeit zu `src/*.py`.

**Ergebnis:** `data/processed/hast_features.csv` — die Eingabe für [`01_kmeans_stationscluster.ipynb`](01_kmeans_stationscluster.ipynb).

## 1 · Setup

In [ ]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 40)
plt.rcParams["figure.facecolor"] = "white"

ROOT = Path.cwd().resolve()
while not (ROOT / "data" / "raw").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
MERGED_DIR = PROCESSED / "real_meters"
MERGED_DIR.mkdir(parents=True, exist_ok=True)
print(f"Rohdaten: {RAW}")

**Zusammenfassung**
- Importiert pandas, numpy, matplotlib
- Sucht automatisch den Projekt-Ordner
- Legt 3 Pfade fest:
    - RAW --> wo die Rohdaten liegen (CSV- und ODS-Exporte) 
    - PROCESSED --> wo Ergebnisse gespeichert werden 
    - MERGED_DIR --> wo die zusemmengeführten Zähler-Dateien (aus Zelle 2) laden 
- Erstellt den MERGED_DIR-Ordner, falls er noch nicht existiert
- Gibt den gefundenen Rohdaten-Pfad aus  

## 2 · Export-Generationen zusammenführen

Pro Zähler gibt es überlappende Exporte in zwei Formaten: CSV (`<Zählernummer>__n.csv` / `<Zählernummer>_2025n.csv`) und ODS (`<Zählernummer>.ods`). Fünf Eigenheiten der Rohdaten werden hier behandelt:

1. **ODS hat nur 6 statt 8 Spalten** — `Energy (kWh)` und `Volume (m³)` fehlen und gelten deshalb als optional.
2. **ODS-Werte sind um Faktor 1000 skaliert** — dort steht z. B. `36000` für 36,0 °C Vorlauf. Wird automatisch erkannt und zurückgerechnet; ohne Korrektur ergäben sich unmögliche Leistungen von 60.000 kW pro HAST (das gesamte Netz hat nur ~2 MW).
3. **Zeitstempel sind minutenversetzt** — dieselbe Stunde steht in der einen Datei als `13:00`, in der anderen als `13:01`. Ohne Abrundung auf die volle Stunde überleben beide die Duplikatentfernung; bei einem Zähler waren dadurch 47 % der Zeilen Dubletten.
4. **Lücken waren unsichtbar** — die Dateien enthielten nur gemessene Stunden. Ein lückenloses Stundenraster macht die Lücken sichtbar und stellt sicher, dass „eine Zeile = eine Stunde" gilt (nötig für die FFT in Zelle 6).
5. **Unmögliche Messwerte** — Plausibilitätsgrenzen setzen Defekte auf NaN, bevor sie in die Kennzahlen laufen.

Alle gefundenen Dateien pro Zähler werden geladen, nach Zeitstempel sortiert und dedupliziert (neuere Generation gewinnt bei Überlappung). Fehlt eine der Pflichtspalten, wird nur dieser eine Zähler übersprungen statt der ganze Lauf abzubrechen — und explizit gelistet, damit das nicht stillschweigend passiert.

In [ ]:
SCHEMA = [
    "Timestamp", "Energy (kWh)", "Volume flow (l/h)", "Power (kW)",
    "Temperature difference (°C)", "Flow temperature (°C)",
    "Return temperature (°C)", "Volume (m³)",
]
OPTIONAL_COLUMNS = ["Energy (kWh)", "Volume (m³)"]
REQUIRED_COLUMNS = [c for c in SCHEMA if c not in OPTIONAL_COLUMNS]
MEASUREMENT_COLUMNS = [c for c in SCHEMA if c != "Timestamp"]

CSV_RE = re.compile(r"^(\d+)_.*\.csv$")
ODS_RE = re.compile(r"^(\d+)\.ods$")

SCALE_FACTOR = 1000.0
SCALE_DETECT_THRESHOLD = 1000.0

PLAUSIBLE_RANGE = {
    "Volume flow (l/h)":           (-1_000, 50_000),
    "Power (kW)":                  (-2_000, 2_000),
    "Temperature difference (°C)": (-60, 100),
    "Flow temperature (°C)":       (-20, 130),
    "Return temperature (°C)":     (-20, 130),
}

def read_export_file(path):
    """Liest eine Export-Datei (CSV oder ODS) im einheitlichen Schema.

    Rückgabe: (DataFrame oder None, Fehlermeldung oder None, Statistik-dict)
    """
    stats = {"rescaled": False, "masked_cells": 0, "invalid_timestamps": 0}
    if path.suffix.lower() == ".csv":
        frame = pd.read_csv(path)
    else:
        frame = pd.read_excel(path, engine="odf")

    missing_required = [c for c in REQUIRED_COLUMNS if c not in frame.columns]
    if missing_required:
        return None, f"{path.name}: fehlende Pflichtspalten {missing_required}", stats

    # Zeitstempel parsen; unlesbare Zeilen fliegen raus
    frame["Timestamp"] = pd.to_datetime(frame["Timestamp"], errors="coerce")
    stats["invalid_timestamps"] = int(frame["Timestamp"].isna().sum())
    frame = frame.dropna(subset=["Timestamp"]).copy()

    for col in OPTIONAL_COLUMNS:
        if col not in frame.columns:
            frame[col] = np.nan
    frame = frame[SCHEMA].copy()
    for col in MEASUREMENT_COLUMNS:
        frame[col] = pd.to_numeric(frame[col], errors="coerce")

    # Erst zurückskalieren, dann auf Plausibilität prüfen - sonst würden
    # die ODS-Werte allesamt als "unmöglich" aussortiert.
    median_flow_temp = frame["Flow temperature (°C)"].abs().median()
    if pd.notna(median_flow_temp) and median_flow_temp > SCALE_DETECT_THRESHOLD:
        frame[MEASUREMENT_COLUMNS] = frame[MEASUREMENT_COLUMNS] / SCALE_FACTOR
        stats["rescaled"] = True

    for col, (low, high) in PLAUSIBLE_RANGE.items():
        outside = (frame[col] < low) | (frame[col] > high)
        stats["masked_cells"] += int(outside.sum())
        frame.loc[outside, col] = np.nan

    return frame, None, stats


meter_files = {}
for f in sorted(RAW.glob("*.csv")):
    m = CSV_RE.match(f.name)
    if m:
        meter_files.setdefault(int(m.group(1)), []).append(f)
for f in sorted(RAW.glob("*.ods")):
    m = ODS_RE.match(f.name)
    if m:
        meter_files.setdefault(int(m.group(1)), []).append(f)
print(f"{sum(len(v) for v in meter_files.values())} Export-Dateien für {len(meter_files)} Zähler gefunden")

for stale in MERGED_DIR.glob("*.csv"):
    stale.unlink()

inventory_rows, skipped, rescaled_files = [], [], []
total_masked = total_invalid_ts = 0
for zid, files in sorted(meter_files.items()):
    frames, problem = [], None
    for path in files:
        frame, error, stats = read_export_file(path)
        if error:
            problem = error
            break
        if stats["rescaled"]:
            rescaled_files.append(path.name)
        total_masked += stats["masked_cells"]
        total_invalid_ts += stats["invalid_timestamps"]
        frames.append(frame)
    if problem:
        skipped.append((zid, problem))
        continue
    
    frames.sort(key=lambda fr: fr["Timestamp"].max())
    merged = pd.concat(frames, ignore_index=True)

    rows_raw = len(merged)
    rows_exact = len(merged.drop_duplicates(subset="Timestamp"))
    merged["Timestamp"] = merged["Timestamp"].dt.floor("h")
    merged = merged.sort_values("Timestamp", kind="stable")
    merged = merged.drop_duplicates(subset="Timestamp", keep="last").reset_index(drop=True)
    rows_measured = len(merged)

    full_index = pd.date_range(merged["Timestamp"].min(),
                               merged["Timestamp"].max(), freq="h")
    merged = (merged.set_index("Timestamp")
                    .reindex(full_index)
                    .rename_axis("Timestamp")
                    .reset_index())

    merged.to_csv(MERGED_DIR / f"{zid}.csv", index=False)
    inventory_rows.append({
        "Zählernummer": zid, "n_files": len(files),
        "n_rows": rows_measured,                    # tatsächlich gemessene Stunden
        "n_hours": len(merged),                     # Länge des Zeitrasters
        "coverage": rows_measured / len(merged),    # Abdeckungsquote
        "overlap_rows": rows_raw - rows_exact,
        "minute_offset_rows": rows_exact - rows_measured,
        "ts_min": merged["Timestamp"].min(), "ts_max": merged["Timestamp"].max(),
    })

inventory = pd.DataFrame(inventory_rows)
print(f"{len(inventory)} Zähler zusammengeführt -> {MERGED_DIR}  (Zeitachse: Lokalzeit)")
if rescaled_files:
    print(f"{len(rescaled_files)} Datei(en) um Faktor {SCALE_FACTOR:.0f} zurückskaliert: "
          f"{', '.join(sorted(rescaled_files))}")
print(f"{total_invalid_ts:,} Zeilen mit unlesbarem Zeitstempel verworfen")
print(f"{total_masked:,} Messzellen als physikalisch unmöglich maskiert")
print(f"{inventory['overlap_rows'].sum():,} Zeilen aus der normalen Exportüberlappung entfernt")
print(f"{inventory['minute_offset_rows'].sum():,} zusätzliche Zeilen durch Minutenversatz entfernt")

affected = inventory.loc[inventory["minute_offset_rows"] > 0].copy()
affected["anteil"] = affected["minute_offset_rows"] / (affected["n_rows"] + affected["minute_offset_rows"])
heavy = affected.loc[affected["anteil"] > 0.05].sort_values("anteil", ascending=False)
print(f"  betroffene Zähler: {len(affected)}, davon >5 % der Zeilen: {len(heavy)}")
for row in heavy.itertuples(index=False):
    print(f"  - {row.Zählernummer}: {row.minute_offset_rows:,} Stunden ({row.anteil*100:.0f} %)")

print(f"\nStundenraster: {inventory['n_hours'].sum():,} Rasterstunden, davon "
      f"{inventory['n_rows'].sum():,} gemessen "
      f"({inventory['n_rows'].sum() / inventory['n_hours'].sum():.0%})")
print(f"  Abdeckung je Zähler: min {inventory['coverage'].min():.0%}, "
      f"Median {inventory['coverage'].median():.0%}, max {inventory['coverage'].max():.0%}")
low_coverage = inventory.loc[inventory["coverage"] < 0.70].sort_values("coverage")
if len(low_coverage):
    print(f"  unter 70 % Abdeckung: {len(low_coverage)} Zähler, schlechteste:")
    for row in low_coverage.head(5).itertuples(index=False):
        print(f"    - {row.Zählernummer}: {row.coverage:.0%}")

if skipped:
    print(f"{len(skipped)} übersprungen (fehlende Pflichtspalten im Export):")
    for zid, reason in skipped:
        print(f"  - {zid}: {reason}")

**Zusammenfassung**
- Legt fest, welche Spalten **Pflicht** sind (Timestamp, Durchfluss, Leistung, ∆T, Volauf, Rücklauf) und welche **optional** (Energy, Volume) - weil die .ods-Dateien die zwei optionalen nicht haben 
- Baut eine kleine Lese-Funktion, die anhand der Dateiendung automatisch die reichtige Lesemethode wählt (CSV oder ODS), prüft ob die Pflichtspalten da sind, und füllt fehlende optionale Spalten mit "leer" (NaN) auf
- Druchsucht data/raw/ nach allen passenden CSV- und ODS-Dateien und gruppiert sie nach Zählernummer 
- Räumt den Zielordner (real_meters) leer, bevor neu reingeschrieben wird - damit keine alten Reste von früheren Läufen übrig bleiben 
- Geht jeden Zähler einzlen durch, lädt alle seine Dateien; fehlt eine von den Pflichtspalten, wird der ganze Zähler übersprungen (mit Grund notiert), statt das Notebook abstürzen zu lassen 
- Hängt die Dateien eines Zählers aneinande, sortiert nach Zeitstempel, entfernt doppelte Zeitpunkte (bei Überlappung gewinnt die neuere Version)
- Speichert das Ergebnis als eine saubere CSV pro Zähler in real_meters 
- Gibt am Ende eine Übersicht aus: wie viele Zähler erforgreich zusammengeführt wurden und welche (mit Grund) übersprungen wurden 

**Changes**
- **Skalierung der ODS-Dateien korrigieren:** 
  Die .ods-Exporte speichern alle Messwerte um Faktor 1000 zu groß (z.B. 36000 statt 36.0 °C Vorlauf)
  Wird über die Vorlauftemperatur automatisch erkannt (Median > 1000) und zurückgerechnet 
  Ohne diese Korrektur kämen unmögliche 60.000 kW pro HAST
  heraus - das ganze Netz hat nur ~2 MW

- **Zeitstempel auf die volle Stunde abrunden:** 
  Dieselbe Stunde steht in der einen Exportgeneration als 13:00, in der anderen als 13:01 
  Ohne Abrunden gelten die als zwei verschiedene Zeitpunkte und überleben beide die Duplikatentfernung -> dieselbe Stunde landet doppelt in der Reihe
  Verfälscht vor allem die FFT in Zelle 6, die "ein Wert = eine Stunde" annimmt
  --> **Zwei Arten von Dubletten werden getrennt gezählt**
  - *Exportüberlappung* (identische Zeitstempel) --> normal, die beiden Generationen decken bewusst denselben Zeitraum ab 
  - *Minutenversatz* (13:00 vs. 13:01) --> der eigentliche Fallstrick, betrifft 3 Zähler stark: 68956353 (47%), 80912193 (31%), 66761022 (17%)

- **Lückenloses Stundenraster:** 
  Nach dem Deduplizieren wird für JEDE Stunde des Zeitraums eine Zeile angelegt
  (`pd.date_range(...)` + `.reindex(...)`), nicht gemessene Stunden werden NaN
  Vorher enthielt die Datei nur gemessene Stunden - eine Lücke war eine fehlende
  Zeile und damit unsichtbar
  --> Damit galt "eine Zeile = eine Stunde" nicht mehr, was die FFT in Zelle 6
      verfälscht (die rechnet mit d=1.0, also genau 1 Stunde Abstand je Zeile)
   - **Drei neue Spalten im inventory:**
     - `n_rows` --> tatsächlich gemessene Stunden (Bedeutung unverändert)
     - `n_hours` --> Länge des Zeitrasters
     - `coverage` --> Abdeckungsquote = n_rows / n_hours

   - **Was dabei herauskam:**
     - 2.558.753 Rasterstunden gesamt, davon nur 1.860.721 gemessen (73 %)
     - Abdeckung je Zähler: min 17 %, Median 73 %, max 99 %
     - 21 Zähler liegen unter 70 %, zwei sogar bei nur 17 %
     - Über ein Viertel aller Messstunden im Netz fehlt - vorher unsichtbar

- **Physikalische Plausibilitätsgrenzen:**
  Werte außerhalb sind kein Messwert, sondern ein Defekt --> werden auf NaN gesetzt
    - Durchfluss    -1.000 bis 50.000 l/h
    - Leistung      -2.000 bis  2.000 kW
    - ΔT               -60 bis    100 °C
    - Vor-/Rücklauf    -20 bis    130 °C
  
  Bewusst weit gefasst: soll kaputte Dateien abfangen, nicht ungewöhnliche aber
  echte Betriebszustände wegwerfen
  
  --> Negative Leistungen bleiben erhalten: bei 72167783 fließt nach der Abtrennung real Wärme rückwärts (bis -14,7 kW), das ist ein Befund und kein Messfehler
  
   --> Ergebnis: nur 1 von 1,86 Mio. Messzellen maskiert. Nach der
      Skalierungskorrektur sind die Daten physikalisch sauber - die Prüfung wirkt vor allem als Absicherung für künftige Exporte
  
  --> Reihenfolge wichtig: erst zurückskalieren, dann prüfen. Andersherum würden alle ODS-Werte als "unmöglich" aussortiert

**Bewusst KEINE Zeitzonen-Umrechnung** (getestet und wieder verworfen):
  Naheliegend wäre Lokalzeit --> UTC, damit die Zeitachse eindeutig wird (im Oktober gibt es 02:00 zweimal, im März gar nicht)
  
  ABER: es kostet Daten, statt welche zu gewinnen
   - Die Exporte enthalten für die doppelte Oktoberstunde nur EINE Ablesung ("no repeated times") --> `ambiguous="infer"` scheitert bei 166 von 198 Dateien. Es gibt also keine zweite Stunde zu retten
   - `ambiguous="NaT"`  --> 405 gemessene Stunden verloren
   - `ambiguous=False`  --> immer noch 240 verloren, weil durch die

      Verschiebung um 1-2 Stunden mehr Zeilen der zwei Exportgenerationen
      im selben Stundenfach landen und dedupliziert werden
  
  --> Zeitstempel bleiben deutsche Lokalzeit wie in den Rohdaten


## 3 · Topologie laden (Adresse, Anschlusswert je Zähler)

In [ ]:
nodes = pd.read_excel(RAW / "Nodes_Edges.ods", sheet_name="Nodes", engine="odf")
nodes["Zählernummer"] = pd.to_numeric(nodes["Zählernummer"], errors="coerce")
print(f"{len(nodes)} HAST in der Topologie")
nodes.head()

**Zusammenfassung**
- Liest die Tabelle "Nodes" aus Nodes_Edges.ods
- Wandelt die Spalte "Zählernummer" in echte Zahlen um
  --> Damit später der Abgleich mit den Zählernummern aus Zelle 2 funtioniert
- Gibt aus, wie viele HAST (Zeilen) in der Topologie gefungen wurden 
- Zeigt die ersten 5 Zeilen zur Kontrolle 

## 4 · Zeitliche Abdeckung

Die Zähler-Historien starten und enden nicht einheitlich (Zähler wurden über die Jahre getauscht/nachgerüstet) — das zeigt sich hier direkt.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(inventory["ts_min"], bins=20, color="#2563EB")
axes[0].set_title("Beginn der Zeitreihe je Zähler")
axes[0].tick_params(axis="x", rotation=30)
axes[1].hist(inventory["n_rows"], bins=20, color="#D97706")
axes[1].set_title("Anzahl Stundenwerte je Zähler")
for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()

**Was macht der Code?**
- Erstellt eine Grafik mit zwei Diagrammen nebeneinander
- **Linkes Diagramm:** Histogramm - wann beginnt die Zeitreihe jedes Zählers
- **Rechtes Diagramm:** Histogramm - wie viele Stunden wurden je Zähler
  tatsächlich gemessen (`n_rows`)
- Achtung: `n_rows` sind die GEMESSENEN Stunden, nicht die Länge des Zeitraums.
  Letztere steht seit dem Stundenraster in `n_hours`, das Verhältnis in `coverage`

**Histogramme erklären**
**Links - zwei klare Ausbaustufen:**
- **49 Zähler** starten im 4. Quartal 2021, **41 Zähler** im 2. Quartal 2023
  --> zwei Rollout-Wellen, keine gleichmäßige Verteilung
- Dazwischen und danach nur Einzelfälle (1-3 pro Quartal), plus 3 Zähler
  schon ab dem 2. Quartal 2021
- Keine Zeitreihe beginnt nach dem 3. Quartal 2024

**Rechts - passend dazu zwei Längen-Gruppen:**
| Gemessene Stunden | Anzahl Zähler | entspricht |
|---|---|---|
| unter 10.000 | 7 | die kurzen Reihen |
| 10.000-20.000 | 49 | die 2023er-Welle |
| 20.000-25.000 | 41 | die 2021er-Welle |
| 30.000-35.000 | 3 | die ODS-Zähler mit Daten bis 2026 |

- Median: 16.124 gemessene Stunden, Spanne von 4.882 bis 32.656

**Praktische Bedeutung**
1. Die Zähler sind unterschiedlich lang beobachtet (0,9 bis 4,7 Jahre).
   Nachgerechnet: beim Clustering verschiebt das die Zuordnung messbar, aber
   moderat - das Verhältnis der Median-Messdauern zwischen den Clustern sinkt
   von 1,75x auf 1,34x, wenn man das messdauerabhängige Merkmal entfernt.
   Die Silhouette ändert sich dabei kaum (0,217 --> 0,224)
2. Die 7 Zähler unter 10.000 Stunden (gut ein Jahr) sind für   saisonale Features (Winter-Rücklauftemperatur, Sommer-Grundlast evtl. zu kurz

## 5 · Beispiel-Zeitreihen ansehen (Sanity-Check)

In [ ]:
sample_ids = inventory.sort_values("n_rows", ascending=False)["Zählernummer"].head(3).tolist()
fig, axes = plt.subplots(len(sample_ids), 1, figsize=(13, 3 * len(sample_ids)))
for ax, zid in zip(axes, sample_ids):
    df = pd.read_csv(MERGED_DIR / f"{zid}.csv", parse_dates=["Timestamp"])
    ax.plot(df["Timestamp"], df["Power (kW)"], linewidth=0.6, color="#2563EB")
    ax.set_title(f"Zähler {zid}: Leistung (kW)")
    ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()

**Was macht der Code**
- Sucht die 3 Zähler mit den meisten gemessenen Stunden (`n_rows` aus Zelle 2)
- Erstellt für jeden ein eigenes Diagramm untereinander
- Lädt die zusammengeführte CSV aus real_meters und plottet die Leistung (kW) über die gesamte Zeitachse
- Reiner Sanity-Check mit dem Auge: sieht die Zusammenführung plausibel aus? Sprünge an den Nahtstellen? Lücken? Unmögliche Werte?

**Die drei gezeigten Zähler**
| Zähler | gemessen | Zeitraum | Abdeckung | ΔT Winter unter Last | max. Leistung |
|---|---|---|---|---|---|
| 68956347 | 32.656 h | 11/2021 - 07/2026 | 79 % | Median 33,0 °C | 31,2 kW |
| 72167783 | 32.654 h | 11/2021 - 07/2026 | 79 % | Median 39,6 °C | 63,4 kW |
| 68956372 | 32.625 h | 11/2021 - 07/2026 | 79 % | Median 17,4 °C | 20,2 kW |

**Was man in den Plots sieht**
- Es sind 3 der 5 ODS-Zähler - kein Zufall: nur die haben Daten bis Juli 2026, alle anderen enden im Juli 2025
- **68956347**: gesunder Jahresgang, Winter hoch / Sommer niedrig, ΔT 33 °C --> so soll eine HAST aussehen
- **68956372**: ebenfalls sauberer Jahresgang, aber mit nur 17 °C deutlich schwächerer Spreizung --> kühlt das Wasser weniger gut aus
- **72167783**: auffällig. Lange Nullstrecken, Aktivität nur in Blöcken.

Kurios: WENN die Station läuft, hat sie mit 39,6 °C die beste Spreizung der drei - aber das 25 %-Quantil liegt bei nur 12,5 °C. Genau diese Zweiteilung (entweder voll oder gar nicht) ist das Warnsignal

## 6 · Features pro HAST berechnen

14 berechnete Kennzahlen pro HAST: Rücklauftemperatur, ΔT unter Last, Sommer-Grundlast (Dauerdurchfluss-/Leckage-Indikator), Trend der Filterverschmutzung (95.-Perzentil-Durchfluss über die Monate), Lastauslastung gegenüber Vertragsleistung, 4–12h-Regelhysterese (FFT-Leistung im Durchfluss), Standby-Verluste, Jahresenergie. Dazu kommen Zählernummer, Adresse, Anschlusswert und die drei Abdeckungsspalten aus Zelle 2 — zusammen 20 Spalten.


In [ ]:
RENAME = {
    "Energy (kWh)": "energy", "Volume flow (l/h)": "flow", "Power (kW)": "power",
    "Temperature difference (°C)": "dt", "Flow temperature (°C)": "vl",
    "Return temperature (°C)": "rt", "Volume (m³)": "volume",
}


def share_of_measured(condition, reference):
    """Anteil erfüllter Bedingungen - bezogen auf die GEMESSENEN Stunden.

    Seit dem Stundenraster aus Zelle 2 enthält jede Reihe auch Leerzeilen
    (NaN) für nicht gemessene Stunden. Ein Vergleich mit NaN ergibt immer
    False, ein schlichtes .mean() würde die Lücken also als "Bedingung
    nicht erfüllt" mitzählen und den Anteil künstlich verwässern - und
    zwar umso stärker, je mehr Lücken ein Zähler hat. Deshalb wird hier
    explizit durch die Anzahl der tatsächlich gemessenen Stunden geteilt.
    """
    n_measured = reference.notna().sum()
    return condition.sum() / n_measured if n_measured else np.nan


def compute_hast_features(raw_df, anschluss_kw=None):
    df = raw_df.rename(columns=RENAME).set_index("Timestamp").sort_index()
    df["month"] = df.index.month
    f = {}
    winter = df["month"].isin([12, 1, 2, 3])
    summer = df["month"].isin([6, 7, 8])
    loaded = df["power"] > 0.5

    f["rt_mean_winter"] = df.loc[winter, "rt"].mean()
    f["rt_p95"] = df["rt"].quantile(0.95)
    f["dt_mean_loaded"] = df.loc[loaded, "dt"].mean()
    f["dt_std_loaded"] = df.loc[loaded, "dt"].std()
    f["rt_above_50_share_loaded"] = share_of_measured(
        df.loc[loaded, "rt"] > 50, df.loc[loaded, "rt"]
    )

    summer_flow = df.loc[summer, "flow"]
    f["summer_flow_share"] = share_of_measured(summer_flow > 5, summer_flow)
    f["summer_flow_baseline"] = summer_flow.median()

    monthly_p95_flow = df["flow"].resample("ME").quantile(0.95).dropna()
    if monthly_p95_flow.size >= 6:
        idx = monthly_p95_flow.index
        x = idx.year * 12 + idx.month
        f["flow_ceiling_slope_per_month"] = np.polyfit(x - x[0], monthly_p95_flow.values, 1)[0]
    else:
        f["flow_ceiling_slope_per_month"] = np.nan

    peak = df["power"].max()
    p95 = df["power"].quantile(0.95)
    if anschluss_kw and anschluss_kw > 0:
        f["peak_load_share"] = peak / anschluss_kw
        f["p95_load_share"] = p95 / anschluss_kw
    else:
        f["peak_load_share"] = np.nan
        f["p95_load_share"] = np.nan

    # Die FFT setzt eine gleichmäßige Zeitachse voraus (d=1.0 = eine Stunde
    # je Zeile). Genau das stellt das Stundenraster aus Zelle 2 sicher.
    # Die Lücken werden nach dem Abziehen des Mittelwerts zu 0 - das
    # entspricht "Durchschnittswert" und fügt kein künstliches Signal ein.
    flow_centered = df["flow"].to_numpy(dtype=float)
    flow_centered = flow_centered - np.nanmean(flow_centered)
    if len(flow_centered) > 0 and np.nanstd(flow_centered) > 0:
        spec = np.abs(np.fft.rfft(np.nan_to_num(flow_centered)))
        freqs = np.fft.rfftfreq(len(flow_centered), d=1.0)
        mask = (freqs >= 1 / 12) & (freqs <= 1 / 4)
        f["flow_short_cycle_power"] = (spec[mask] ** 2).sum() / (spec ** 2).sum()
    else:
        f["flow_short_cycle_power"] = np.nan

    f["idle_share"] = share_of_measured(df["power"] < 0.2, df["power"])

    # Manche Exporte (die .ods-Dateien) liefern keine Energy-Spalte -
    # dann bleiben diese zwei Features NaN statt einen Fehler zu werfen.
    energy_values = df["energy"].dropna()
    if len(energy_values) >= 2:
        energy_year = energy_values.iloc[-1] - energy_values.iloc[0]
    else:
        energy_year = np.nan
    f["energy_kwh_year"] = energy_year
    f["full_load_hours"] = energy_year / max(peak, 0.1) if pd.notna(energy_year) else np.nan
    return f


rows = []
for csv_path in sorted(MERGED_DIR.glob("*.csv")):
    zid = int(csv_path.stem)
    raw_df = pd.read_csv(csv_path, parse_dates=["Timestamp"])
    match = nodes.loc[nodes["Zählernummer"] == zid]
    kw = float(match["Anschlusswert"].iloc[0]) if len(match) else None
    feats = compute_hast_features(raw_df, anschluss_kw=kw)
    feats["Zählernummer"] = zid
    if len(match):
        feats["address"] = match["Straße"].iloc[0]
        feats["anschlusswert_kw"] = kw
    rows.append(feats)

features = pd.DataFrame(rows)
# Abdeckung aus Zelle 2 mitführen: sagt für jede Station, auf wie vielen
# echten Messstunden ihre Kennzahlen überhaupt beruhen.
features = features.merge(
    inventory[["Zählernummer", "coverage", "n_rows", "n_hours"]],
    on="Zählernummer", how="left",
)
print(f"Features für {len(features)} HAST berechnet, {features.shape[1]} Spalten")
features.head()

**Das Grundproblem**
Wir haben 101 Dateien. Jede enthält für einen Zähler zehntausende Zeilen - eine Zeile pro Stunde, über mehrere Jahre.
Damit kann man nicht clustern. K-Means braucht eine Tabelle mit **einer Zeile pro Station**. also muss jede Zeitreihe zu ein paar Zahlen zusammengedampft werden. 
--> 101 Dateien zu einer Tabelle mit 101 Zeilen

**Ablauf**
Teil 1: Eine Funktion wird definiert (compute_hast_features)
--> Gib mir die Zeitreihe eines Zählers - ich gebe dir 14 Zahlen zurück, die diesen Zähler beschreiben.
Teil 2: Eine Schleife ganz unten. Geht alle 101 Dateien durch 

Schritt 1: Spalten umbennen:
  Aus "Temperatur difference" wird dt
  --> kürzerer Code

Schritt 2: Zeilen markieren:
  Für jede der 30.000 Zeilen wird notiert
  - Ist das ein Wintermonat? (Dez, Jan, Feb, Mär)
  - Ist das ein Sommermonat? (Jun, Jul, Aug)
  - Lief die Station gerade? (Leistung über 0,5 kW)

Schritt 3: Zusammenrechnen:
  Jetzt kommen die eigentlichen 14 Zahlen. Jede ist eine simple Rechnung über eine Teilmenge der Zeilen. Zum Beispiel:

| Kennzahl | Rechnung |
|---|---|
| `rt_mean_winter` | Nimm nur die Winter-Zeilen → Mittelwert der Rücklauftemperatur |
| `summer_flow_baseline` | Nimm nur die Sommer-Zeilen → Median des Durchflusses |
| `dt_mean_loaded` | Nimm nur die Zeilen, wo die Station lief → Mittelwert von ΔT |
| `idle_share` | Zähle Zeilen mit fast null Leistung → teile durch die GEMESSENEN Stunden |
  
**Changes**
- **Neue Hilfsfunktion `share_of_measured()`:**
  Seit dem Stundenraster gibt es NaN-Zeilen für nicht gemessene Stunden
  Ein Vergleich mit NaN ergibt immer False --> ein schlichtes `.mean()` würde
  die Lücken als "Bedingung nicht erfüllt" mitzählen
  Ergebnis wäre exakt `korrekter Wert x Abdeckung` gewesen - beim Zähler mit
  17 % Abdeckung also nur noch 17 % des richtigen Werts
  --> Wird jetzt für `idle_share`, `summer_flow_share` und
      `rt_above_50_share_loaded` verwendet

- **Abdeckung wandert in die Feature-Tabelle:**
  `coverage`, `n_rows`, `n_hours` werden aus dem inventory gemerged
  --> sagt für jede Station, auf wie vielen echten Messstunden ihre Kennzahlen
      überhaupt beruhen
  --> 20 statt 17 Spalten

- **Effekt auf `flow_short_cycle_power`** (drittstärkster Treiber im Clustering):
    - Median über alle HAST: 0,1866 --> 0,1742 (-4,3 %)
    - 8 Stationen lagen um 20-34 % daneben, am stärksten 80912182 (0,182 --> 0,120)
    - Überraschung: die Verzerrung hängt NICHT an der Lückenmenge
      (Korrelation nur +0,23). Entscheidend ist die Verteilung - viele kleine
      verstreute Lücken stauchen die Zeitachse durchgehend, wenige große
      Blöcke verschieben sie nur einmal


## 7 · Datenqualität der Features

In [ ]:
missing = features.drop(columns=["Zählernummer", "address"], errors="ignore").isna().mean().sort_values(ascending=False) * 100
missing = missing[missing > 0]
print("Anteil fehlender Werte pro Feature (nur >0%):")
display(missing.round(1).to_frame("% fehlend"))

**Was macht der Code** 
- Wirft die zwei Namensschild-Spalten raus (Zählernummer, address) - das sind keine Messwerte 
- `.isna()`--> ersetzt jede Zelle durch True (leer) / False (hat einen Wert)
- `.mean()`--> Druchschnitt pro Spalte. Trick dabei: True zählt als 1, False als 0, der Durchschnitt ist also genau der Anteil der leeren Zellen 
- `* 100`und sortieren --> Prozent, das Schlimmste oben 
- `missing[missing > 0]` --> blendet alle Kennzahlen ohne Lücken aus 

**Ergebnis und was dahinter steckt**
| Kennzahl | fehlend | Grund |
|---|---|---|
| `anschlusswert_kw` | 15,8 % | 16 Zähler stehen nicht in Nodes_Edges.ods |
| `peak_load_share` | 15,8 % | rechnet mit dem Anschlusswert --> erbt die Lücke |
| `p95_load_share` | 15,8 % | dito |
| `dt_mean_loaded` | 5,9 % | 6 Stationen hatten nie Last über 0,5 kW |
| `dt_std_loaded` | 5,9 % | dito |
| `flow_short_cycle_power` | 2,0 % | 2 Zähler ohne Schwankung im Durchfluss, FFT nicht möglich |
| `rt_above_50_share_loaded` | 2,0 % | dito |

**Warum das fürs Clustering wichtig ist**
- K-Means kann mit leeren Werten nicht rechnen --> Notebook 01 ersetzt sie durch den Median aller anderen Stationen
- Nachgemessen: die Auswirkung ist geringer als befürchtet. `peak_load_share`und `p95_load_share` tragen mit einer Trennschärfe von nur 0,04 praktisch nichts zur Clusterbildung bei - ausgerechnet die beiden Merkmale mit den meisten Lücken sind die unwichtigsten
- Insgesamt sind 46 von 1.010 Zellen imputiert (4,6 %), betroffen sind 16 von 101 Stationen
- Die Zelle bleibt trotzdem wichtig: sie zeigt, auf welcher Datenbasis jede Kennzahl überhaupt steht

  Vertrauen das Clustering-Ergebnis pro Station verdient

## 8 · Verteilungen ausgewählter Features

In [ ]:
# Pro Kennzahl: sprechender Titel, X-Achsenbeschriftung mit Einheit und ein
# Faktor, um Anteile (0-1) als Prozent darzustellen.
PLOT_SPECS = [
    ("rt_mean_winter",         "Rücklauftemperatur im Winter",     "°C",                              1),
    ("dt_mean_loaded",         "Spreizung ΔT unter Last",          "°C",                              1),
    ("peak_load_share",        "Auslastung der Vertragsleistung",  "Spitzenlast / Anschlusswert",     1),
    ("flow_short_cycle_power", "Kurzzyklen im Durchfluss (4–12 h)", "Anteil an der Gesamtvarianz (%)", 100),
    ("summer_flow_baseline",   "Sommer-Grundströmung",             "Median-Durchfluss Jun–Aug (l/h)", 1),
    ("idle_share",             "Standby-Anteil",                   "Stunden mit < 0,2 kW (%)",        100),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, (col, title, xlabel, scale) in zip(axes.flat, PLOT_SPECS):
    values = features[col].dropna() * scale
    ax.hist(values, bins=20, color="#6B7D2A", edgecolor="white")
    # n steht dabei, weil dropna() je Kennzahl unterschiedlich viele
    # Stationen übrig lässt (fehlender Anschlusswert, nie unter Last, ...)
    ax.set_title(f"{title}\n{len(values)} von {len(features)} HAST", fontsize=11)
    ax.set_xlabel(xlabel, fontsize=9)
    ax.set_ylabel("Anzahl HAST", fontsize=9)
    ax.grid(axis="y", color="#E2E8F0", linewidth=0.7)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)

fig.suptitle("Verteilung der Kennzahlen über alle HAST", fontweight="bold", fontsize=13)
fig.tight_layout()
plt.show()

**Was ein Histogramm hier zeigt**
- NICHT den Zeitverlauf, sondern die Verteilung über alle 101 Stationen
- X-Achse: Wertebereich der Kennzahl, in 20 gleich breite Fächer geteilt
- Y-Achse: wie viele Stationen in dieses Fach fallen

**Warum genau diese 6?**
Eine pro Fehlerfamilie, nicht einfach die ersten sechs:

| Kennzahl | Fehlerbild dahinter |
|---|---|
| `rt_mean_winter` | schlechte Auskühlung |
| `dt_mean_loaded` | schlechter Wärmeübergang |
| `peak_load_share` | überdimensionierter Vertrag |
| `flow_short_cycle_power` | schwingende Regelung |
| `summer_flow_baseline` | Dauerströmung / Bypass |
| `idle_share` | Standby-Verhalten |

**Wonach man beim Draufschauen sucht**
Eigentliche Frage dieser Zelle: taugt die Kennzahl überhaupt zum Clustern?
- **Ein einzelner Hügel** --> alle Stationen ähnlich, das Merkmal trennt nichts
- **Zwei oder mehr Hügel** --> es gibt echte Gruppen im Netz, darauf kann
  K-Means aufsetzen
- **Alles links gestapelt, ein paar Ausreißer weit rechts** --> typisch für
  Fehlerkennzahlen. Die rechts außen sind die interessanten Fälle

**Stolperstein: unterschiedliche Grundgesamtheit je Feld**
- Durch `.dropna()` zeigt nicht jedes Feld gleich viele Stationen:
  `rt_mean_winter` alle 101, `dt_mean_loaded` nur 95, `peak_load_share` nur 85
- Deshalb steht die Anzahl jetzt in jedem Titel ("85 von 101 HAST")
- Balkenhöhen zwischen den Feldern sind also nur bedingt vergleichbar

## 9 · Korrelationen zwischen Features

In [ ]:
numeric_cols = features.select_dtypes(include=[np.number]).drop(columns=["Zählernummer"], errors="ignore")
corr = numeric_cols.corr()
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=90, fontsize=8)
ax.set_yticks(range(len(corr.columns)))
ax.set_yticklabels(corr.columns, fontsize=8)
fig.colorbar(im, shrink=0.8)
fig.tight_layout()
plt.show()

**Wie man das Bild liest**
| Farbe | Bedeutung |
|---|---|
| kräftig rot (+1) | beide steigen gemeinsam |
| weiß (0) | kein linearer Zusammenhang |
| kräftig blau (-1) | eine steigt, die andere fällt |

- Die **Diagonale ist immer knallrot** - jede Kennzahl korreliert mit sich selbst perfekt
- Das Bild ist **symmetrisch** - oben rechts spiegelt unten links

**Gemessene Korrelationen (statt vermuteter)**
| Paar | Wert |
|---|---|
| `rt_mean_winter` <-> `dt_mean_loaded` | **0,791** |
| `rt_mean_winter` <-> `flow_short_cycle_power` | 0,477 |
| `peak_load_share` <-> `p95_load_share` | 0,436 |
| `summer_flow_baseline` <-> `full_load_hours` | 0,407 |

- Die starke Doppelung sitzt bei **Rücklauftemperatur <-> Spreizung** (0,79), nicht bei den Lastquoten wie zunächst vermutet. Physikalisch logisch: bei konstantem Vorlauf ist hohe Rücklauftemperatur dasselbe wie niedrige Spreizung
- Das sind ausgerechnet die zwei stärksten Clustering-Treiber
  --> die Auskühlqualität zählt im Clustering doppelt

**Neu in der Matrix: coverage, n_rows, n_hours**
- Seit Zelle 6 stehen diese drei Spalten mit in der Tabelle und erscheinen deshalb auch in der Korrelationsmatrix
- Damit lässt sich direkt ablesen, ob eine Kennzahl mit der MESSDAUER korreliert statt mit dem Betriebsverhalten
- Besonders relevant für `full_load_hours` und `energy_kwh_year`: die summieren über die ganze Reihe statt pro Jahr, sind also messdauerabhängig

**Zwei Einschränkungen**
- **Nur lineare Zusammenhänge:** `.corr()` nutzt Pearson. Zwei Kennzahlen können stark zusammenhängen (z.B. U-förmig) und trotzdem nahe 0 anzeigen
- **Unterschiedliche Grundgesamtheiten:** bei fehlenden Werten rechnet pandas jedes Paar nur über die Zeilen, wo beide vorhanden sind. Manche Felder der Matrix beruhen also auf weniger Stationen als andere - steht aber nirgends dran


## 10 · Ergebnis speichern

In [ ]:
out_path = PROCESSED / "hast_features.csv"
features.to_csv(out_path, index=False)
print(f"{len(features)} HAST-Profile gespeichert -> {out_path}")

## Zusammenfassung

Diese Feature-Tabelle (`data/processed/hast_features.csv`) ist die Grundlage für das Clustering in [`01_kmeans_stationscluster.ipynb`](01_kmeans_stationscluster.ipynb).